# قِس قانون التوسّع بنفسك

**قانون التوسّع نتيجة قياس، وليس ثابتاً** · معالج رسوميات · ~45 دقيقة · Colab

لديك ميزانية حوسبة لم تدرّب عندها أيّ نموذج من قبل، وهي لا تكفي إلا لنموذج واحد. ما الحجم المناسب لهذا النموذج؟ وكم من البيانات يجب أن يقرأ؟ في الأدبيات جواب مشهور: نحو 20 رمزاً تدريبياً لكل معامل. لكنّ هذا الرقم جاء من قياس، على نماذج غيرك وبياناته ومُرمِّزه. في هذا الدفتر تقيس رقمك أنت. تدرّب بضع عشرات من النماذج اللغوية الصغيرة التي تقرأ النص بايتاً بايتاً، ثم توفّق قوانين قوة على أفضلها. بعد ذلك تستخدم هذا التوفيق لتختار نموذجاً عند ميزانية أكبر بـ{{scale.holdout_factor}} مرات من كل ما درّبته، وتتوقّع خسارته قبل أن تدرّبه. وفي إعدادك هذا، يقع أفضل تقسيم عند {{run.metrics.tokens_per_param}} رمزاً لكل معامل.

### الهدف

أن تتوقّع خسارة نموذج لم تدرّبه بعد، عند ميزانية أكبر بـ{{scale.holdout_factor}} مرات من أيّ نقطة دخلت في التوفيق، ثم تدرّبه فعلاً، فيأتي الفرق بين التوقّع والقياس ضمن حدّ الخطأ المعلن.

### الأوراق وراء هذه الورشة

- [training-compute-optimal-large-language-models](https://azimuth.plus/ar/paper/training-compute-optimal-large-language-models) — طريقة منحنيات الحوسبة الثابتة، وسطح الخسارة المعلَمي، ونتيجة الـ20 رمزاً لكل معامل التي نقارن بها قياسنا
- [empirical-scaling-laws-for-language-models](https://azimuth.plus/ar/paper/empirical-scaling-laws-for-language-models) — فكرة أنّ الخسارة تتبع قوانين قوة في المعاملات والبيانات والحوسبة، وطريقة عدّ المعاملات دون جداول التضمين

> احفظ نسخة في Drive قبل أن تبدأ (ملف ← حفظ نسخة في Drive). التعديلات على الأصل لا تُحفظ.

## الإعداد

`PROFILE` هو المقبض الوحيد للحجم. المستوى المجاني هو الافتراضي ويعمل داخل حدود Colab المجانية.

In [ ]:
SLUG = "fit-your-own-scaling-law"
LANG = "ar"
PROFILE = "free"  # free | a100

# Colab defaults to inline figures; CI does not. Being explicit means the
# captured plot on the site and the plot you see are produced the same way.
%matplotlib inline

# The shim lives in this repository, not on PyPI: a workshop should never
# depend on a package index staying up.
import os
import subprocess
import sys
from pathlib import Path

# Guarded three ways. Colab users re-run the setup cell constantly, and
# a contributor may already be sitting inside a checkout — the first
# Windows run of this notebook cloned the repository into its own
# generated/notebooks/ directory because neither case was handled.
# The hash of the code.py THESE CELLS were built from. setup()
# compares it with the code.py it finds on disk: if a notebook is
# older than its source, every number below describes code that is
# not the code anyone is reading. The printed `code ·` line was
# taken from disk and so could not catch this by itself.
os.environ['AZIMUTH_NOTEBOOK_CODEHASH'] = '2abd3bf077e0b191'

REPO = 'azimuth-workshops'
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'shim' / 'azimuth_nb').is_dir()), None)
if root is None:
    if not Path(REPO).exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', 'stable',
             'https://github.com/MotazSabri/azimuth-workshops.git', REPO],
            check=True,
        )
    root = (here / REPO).resolve()
print('workshop root:', root)

# Absolute, so nothing depends on the working directory. Dropping the
# %cd this cell used to do also means the notebook stops caring where
# it was opened from.
sys.path.insert(0, str(root / 'shim'))
_ = os.environ.setdefault('AZIMUTH_DATA_DIR', str(root / 'data'))

In [ ]:
# Declared by this workshop (dependencies: in workshop.yaml).
# torch is NOT installed here — it is asserted, because a second
# torch over Colab's own will not match the driver.
%pip install -q arabic-reshaper==3.0.0 python-bidi==0.4.2

في عام 2020 قاس <bdi dir="ltr">Kaplan</bdi> وزملاؤه كيف تنخفض <Term en="Loss">خسارة</Term> النموذج اللغوي كلما زادت المعاملات والبيانات والحوسبة. جاءت كل علاقة من هذه العلاقات على شكل <Term en="Power Law">قانون قوة</Term>. وكانت نصيحتهم: إذا كبرت الميزانية، فأنفق معظم الزيادة على تكبير النموذج. في عام 2022 أعادت ورقة <bdi dir="ltr">Chinchilla</bdi> القياس بعناية أكبر، فوصلت إلى جواب مختلف. الجواب الجديد: كبّر المعاملات و<Term en="Training Tokens">رموز التدريب</Term> معاً، بنحو 20 رمزاً لكل معامل.

لم يُستنتج أيٌّ من الجوابين من مبادئ أولى. كلاهما جاء من تدريب نماذج كثيرة، ثم رسم نتائجها، ثم توفيق منحنيات عليها. وهذا يعني أنك تستطيع أن تفعل الشيء نفسه على معالج رسومي واحد. الأرقام التي ستحصل عليها ستكون أرقامك أنت، وهذا بالضبط ما يريد هذا الدفتر أن يريك إياه.

> **الورقة** · [training-compute-optimal-large-language-models](https://azimuth.plus/ar/paper/training-compute-optimal-large-language-models) — طريقة منحنيات الحوسبة الثابتة، وسطح الخسارة المعلَمي، ونتيجة الـ20 رمزاً لكل معامل التي نقارن بها قياسنا
>
> يصف القسم الثالث من الورقة الطرق الثلاث. ينفّذ هذا الدفتر الطريقة الثانية، أي منحنيات الحوسبة الثابتة، والطريقة الثالثة، أي سطح الخسارة المعلَمي، وكلتاهما على التشغيلات نفسها.

> **الورقة** · [empirical-scaling-laws-for-language-models](https://azimuth.plus/ar/paper/empirical-scaling-laws-for-language-models) — فكرة أنّ الخسارة تتبع قوانين قوة في المعاملات والبيانات والحوسبة، وطريقة عدّ المعاملات دون جداول التضمين
>
> عدّ <bdi dir="ltr">Kaplan</bdi> وزملاؤه المعاملات دون جداول التضمين. ويبيّن التمرين الأخير ما يفعله هذا الاختيار بالأُسّ.

_تحقّق من بصمة الشيفرة، ومن أنّ معالجاً رسومياً متّصل. فالشبكة تحتاج إليه._

In [ ]:
import azimuth_nb as azimuth

env = azimuth.setup(SLUG, lang=LANG, profile=PROFILE)

تقرأ النماذج البايتات الخام كما هي. لذلك تتكوّن المفردات من 256 رمزاً فقط، ولا يوجد مُرمِّز نحتاج إلى تدريبه أو إلى الاختلاف حوله. وكل خسارة تُقاس بوحدة نات لكل بايت، على نص لم تتدرّب عليه النماذج أبداً.

أول رقم يستحقّ الانتباه هو خط الأساس الأحادي. وهو خسارة نموذج يتجاهل السياق تماماً، ويتوقّع كل بايت من تكراره العام في النص. كل نموذج مدرَّب يجب أن يتفوّق على هذا الخط بفارق واضح. فإذا لم يتفوّق عليه نموذج ما، فهذا النموذج لم يتعلّم شيئاً. وأيّ منحنى نوفّقه عبر نقطة كهذه هو منحنى يمرّ عبر ضجيج.

_يقع خط الأساس قرب 3 نات لكل بايت. احفظ هذا الرقم، فكل نموذج مدرَّب يجب أن ينتهي أدنى منه بكثير._

In [ ]:
import math

import numpy as np
import torch


def tr(en, ar):
    """Pick the printed string for the notebook's language."""
    return ar if env.lang == "ar" else en


cfg = env.cfg
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(cfg["seed"])

raw = np.frombuffer(env.assets["tinystories-v2-head.txt"].read_bytes(), dtype=np.uint8)
raw = raw[: int(cfg["corpus_bytes"])]
n_val = int(cfg["val_bytes"])
val_bytes, train_bytes = raw[:n_val], raw[n_val:]

# The zero-skill reference: predict every byte from its overall frequency.
counts = np.bincount(train_bytes, minlength=256).astype(np.float64)
probs = counts[counts > 0] / counts.sum()
unigram_loss = round(float(-(probs * np.log(probs)).sum()), 4)

# Cut the training bytes into non-overlapping windows and shuffle them ONCE.
# Every run reads a prefix of this same order, so no run sees a byte twice and
# all runs at all sizes see the same data in the same sequence.
ctx = int(cfg["context"])
n_chunks = len(train_bytes) // (ctx + 1)
train_chunks = (
    torch.from_numpy(train_bytes[: n_chunks * (ctx + 1)].copy()).view(n_chunks, ctx + 1).to(device)
)
order = torch.randperm(n_chunks, generator=torch.Generator().manual_seed(cfg["seed"])).to(device)
max_train_tokens = n_chunks * ctx

n_val_chunks = len(val_bytes) // (ctx + 1)
val_chunks = (
    torch.from_numpy(val_bytes[: n_val_chunks * (ctx + 1)].copy())
    .view(n_val_chunks, ctx + 1)
    .to(device)
)

print(
    tr(
        f"device {device} · train {len(train_bytes) / 1e6:.1f} MB · val {len(val_bytes) / 1e6:.2f} MB",
        f"الجهاز {device} · بيانات التدريب {len(train_bytes) / 1e6:.1f} ميغابايت · التحقق {len(val_bytes) / 1e6:.2f} ميغابايت",
    )
)
print(
    tr(
        f"unigram baseline: {unigram_loss:.3f} nats/byte",
        f"خط الأساس الأحادي: {unigram_loss:.3f} نات لكل بايت",
    )
)
print(train_bytes[:300].tobytes().decode("utf-8", errors="replace"))

قانون التوسّع يقارن بين نماذج بأحجام مختلفة، لذلك نحتاج إلى عائلة كاملة منها. المجموعة أدناه تُكبّر العرض والعمق معاً، من بضعة آلاف من <Term en="Parameter Count">المعاملات</Term> إلى بضعة ملايين. وبجانب كل شكل عددان للمعاملات. الأول هو العدد الكلي، كما عدّتها ورقة <bdi dir="ltr">Chinchilla</bdi>. والثاني هو العدد بعد حذف جداول التضمين، كما عدّها <bdi dir="ltr">Kaplan</bdi>. نستخدم العدد الكلي الآن، ونعود إلى الفرق بين العددين في التمرين الأخير.

_انظر إلى أصغر الأشكال: جداول التضمين فيها تمثّل جزءاً كبيراً من عددها الكلي._

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.heads = heads
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        b, t, d = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(d, dim=2)
        q, k, v = (z.view(b, t, self.heads, d // self.heads).transpose(1, 2) for z in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(y.transpose(1, 2).reshape(b, t, d))
        return x + self.mlp(self.ln2(x))


class TinyLM(nn.Module):
    """A byte-level GPT. The output layer shares weights with the input embedding."""

    def __init__(self, n_layer, d_model):
        super().__init__()
        self.tok = nn.Embedding(256, d_model)
        self.pos = nn.Embedding(ctx, d_model)
        self.blocks = nn.ModuleList(Block(d_model, d_model // 16) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(d_model)

    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for blk in self.blocks:
            x = blk(x)
        return self.ln_f(x) @ self.tok.weight.T


def count_params(n_layer, d_model):
    """Two bookkeeping conventions for the same network."""
    block = 12 * d_model * d_model + 13 * d_model  # attention + MLP + norms
    embedding = 256 * d_model + ctx * d_model  # token + position tables
    non_embedding = n_layer * block + 2 * d_model  # + the final norm
    return {"total": non_embedding + embedding, "non_embedding": non_embedding}


# A pool of shapes, close to continuous in size. Width and depth grow together,
# kept within a band of width-to-depth ratios.
ladder = [
    (n_layer, d_model)
    for n_layer in range(1, cfg["max_layers"] + 1)
    for d_model in range(16, cfg["max_width"] + 1, 16)
    if cfg["aspect_min"] <= d_model / n_layer <= cfg["aspect_max"]
]
sizes = {s: count_params(*s) for s in ladder}
ladder.sort(key=lambda s: sizes[s]["total"])

# The formula must agree with the network itself.
for shape in (ladder[0], ladder[-1]):
    real = sum(p.numel() for p in TinyLM(*shape).parameters())
    assert real == sizes[shape]["total"], (shape, real, sizes[shape])


def nearest_shape(n_target):
    return min(ladder, key=lambda s: abs(math.log(sizes[s]["total"] / n_target)))


print(
    tr(
        f"{len(ladder)} shapes, from {sizes[ladder[0]]['total']:,} to {sizes[ladder[-1]]['total']:,} parameters",
        f"{len(ladder)} شكلاً، من {sizes[ladder[0]]['total']:,} إلى {sizes[ladder[-1]]['total']:,} معاملاً",
    )
)
print(
    tr(
        "layers  width   params (total)   non-embedding",
        "الطبقات  العرض   المعاملات (الكلّ)   دون التضمين",
    )
)
for shape in ladder[:: max(1, len(ladder) // 8)]:
    n = sizes[shape]
    print(f"{shape[0]:>6}  {shape[1]:>5}   {n['total']:>14,}   {n['non_embedding']:>13,}")

تدريب محوّل فيه <bdi dir="ltr">N</bdi> معاملاً على <bdi dir="ltr">D</bdi> رمزاً يكلّف تقريباً <bdi dir="ltr">C ≈ 6·N·D</bdi> من <Term en="Training FLOPs">عمليات الفاصلة العائمة</Term>. ومصدر الرقم 6: عمليتان لكل معامل ولكل رمز في المرور الأمامي، وأربع عمليات في المرور الخلفي. هذا التقريب هو ما يسمح لنا بالمقايضة بين <bdi dir="ltr">N</bdi> و<bdi dir="ltr">D</bdi> عند <Term en="Compute Budget">ميزانية حوسبة</Term> ثابتة.

في المدرِّب تفصيل صغير، أهميته أكبر مما يبدو. جدول معدّل التعلّم في كل تشغيل يمتدّ على طول ذلك التشغيل نفسه، لا أكثر. أشار مؤلفو <bdi dir="ltr">Chinchilla</bdi> إلى أنّ ضبط الجدول على تشغيل أطول كان أحد أسباب ميل القانون الأقدم إلى النماذج الكبيرة. فالتشغيلات القصيرة لم يكتمل فيها خفض معدّل التعلّم. ولذلك بدت نتائجها أسوأ من حقيقتها.

_تشغيل صغير واحد قبل الشبكة. ويجب أن تكون خسارته أدنى من خط الأساس الأحادي منذ الآن._

In [ ]:
import time


def train_run(shape, tokens, seed):
    """Train one model on exactly `tokens` fresh tokens; return its validation loss.

    The cosine schedule is stretched to this run's own length. That detail is the
    one the Chinchilla paper singles out: a schedule set for a longer run leaves
    a shorter run undertrained, and the scaling law inherits the error.
    """
    batch = int(cfg["batch"])
    steps = max(1, round(tokens / (batch * ctx)))
    assert steps * batch <= n_chunks, "run would repeat data; lower the budget"
    torch.manual_seed(seed)
    model = TinyLM(*shape).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], betas=(0.9, 0.95), weight_decay=0.1)
    warmup = max(1, int(0.05 * steps))

    def lr_at(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, steps - warmup)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    model.train()
    for step in range(steps):
        rows = train_chunks[order[step * batch : (step + 1) * batch]].long()
        x, y = rows[:, :-1], rows[:, 1:]
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            loss = F.cross_entropy(model(x).float().view(-1, 256), y.reshape(-1))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
    return evaluate(model), steps * batch * ctx


@torch.no_grad()
def evaluate(model):
    model.eval()
    total, n = 0.0, 0
    for i in range(0, len(val_chunks), 256):
        rows = val_chunks[i : i + 256].long()
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            logits = model(rows[:, :-1]).float()
        total += F.cross_entropy(
            logits.reshape(-1, 256), rows[:, 1:].reshape(-1), reduction="sum"
        ).item()
        n += rows[:, 1:].numel()
    return total / n


# One short run to make sure the loop learns before the grid spends real time.
t0 = time.time()
probe_loss, _ = train_run(ladder[0], int(cfg["min_tokens"]), cfg["seed"])
probe_loss = round(probe_loss, 4)
print(
    tr(
        f"probe run: loss {probe_loss:.3f} nats/byte (unigram {unigram_loss:.3f}) in {time.time() - t0:.1f}s",
        f"تشغيل تجريبي: الخسارة {probe_loss:.3f} نات لكل بايت (الأحادي {unigram_loss:.3f}) خلال {time.time() - t0:.1f} ث",
    )
)

هنا قلب الطريقة. اختر ميزانية حوسبة <bdi dir="ltr">C</bdi>. ثم درّب عدّة نماذج تنفق كلها <bdi dir="ltr">C</bdi> بالضبط: من نموذج كبير يقرأ رموزاً قليلة، إلى نموذج صغير يقرأ رموزاً كثيرة. بعد ذلك ارسم خسائرها مقابل أحجامها. الناتج هو <Term en="IsoFLOP Curve">منحنى الحوسبة الثابتة</Term> (<bdi dir="ltr">IsoFLOP</bdi>)، وأدنى نقطة فيه هي أفضل حجم لتلك الميزانية. وتكرّر ذلك لعدّة ميزانيات.

نختار طرق التقسيم بحسب نسبة الرموز إلى المعاملات، <bdi dir="ltr">D / N</bdi>. ونوزّع هذه النسب على مدى واسع عن قصد. فنحن لا نعرف مسبقاً أين يقع أفضل تقسيم. والطريقة لا تنجح إلا إذا وُجدت، في كل ميزانية، تشغيلات أسوأ من أفضل تشغيل على جانبيه كليهما.

_تحصل كل ميزانية على عدّة أحجام. وتغيب الأحجام الكبرى عن الميزانيات الصغيرة، لأنها ستقرأ هناك رموزاً أقلّ من أن تتعلّم منها._

In [ ]:
# Compute budgets, spaced by a constant factor. Within a budget C, each run
# splits the same compute differently: N parameters trained on D = C / (6 N)
# tokens. The splits are chosen by tokens-per-parameter ratio, D / N, over a
# wide range, because we do not know in advance where the best split lies.
budgets = [cfg["budget_min"] * cfg["budget_factor"] ** i for i in range(cfg["n_budgets"])]

plan = []
for C in budgets:
    chosen = set()
    for ratio in cfg["ratios"]:
        shape = nearest_shape(math.sqrt(C / (6 * ratio)))
        tokens = C / (6 * sizes[shape]["total"])
        if shape not in chosen and cfg["min_tokens"] <= tokens <= max_train_tokens:
            chosen.add(shape)
            plan.append({"C": C, "shape": shape, "tokens": tokens})

planned_tokens = sum(p["tokens"] for p in plan)
print(
    tr(
        f"{len(plan)} runs across {len(budgets)} budgets · {planned_tokens / 1e6:.0f}M tokens in total",
        f"{len(plan)} تشغيلاً على {len(budgets)} ميزانيات · {planned_tokens / 1e6:.0f} مليون رمز إجمالاً",
    )
)
for C in budgets:
    row = [p for p in plan if p["C"] == C]
    shown = ", ".join(
        f"{sizes[p['shape']]['total'] / 1e3:.0f}k" for p in sorted(row, key=lambda p: p["tokens"])
    )
    print(f"  C = {C:.1e} FLOPs  →  N = {shown}")

> **الحجم** — يدرّب المستوى المجاني عبر {{scale.n_budgets}} ميزانيات، وكل ميزانية تساوي السابقة مضروبةً في {{scale.budget_factor}}. يُستبعَد أيّ تقسيم يعطي التشغيل أقلّ من {{scale.min_tokens}} رمزاً. والسبب أنّ التشغيل ذا الخطوات القليلة يقيس المُحسِّن، لا النموذج. ويُستبعَد كذلك أيّ تقسيم يحتاج رموزاً أكثر مما في النص. فلا يقرأ أيّ تشغيل البايت نفسه مرتين، وبذلك لا تستطيع البيانات المكرّرة أن تحرف المنحنيات.

_أطول خلية في الدفتر. اقرأ الجدول ميزانيةً ميزانية: يجب أن تنخفض الخسارة ثم ترتفع من جديد كلما كبر <bdi dir="ltr">N</bdi>._

In [ ]:
runs = []
t_grid = time.time()
for i, p in enumerate(plan):
    loss, tokens = train_run(p["shape"], p["tokens"], cfg["seed"] + i)
    n = sizes[p["shape"]]
    runs.append(
        {
            "C": p["C"],
            "shape": p["shape"],
            "N": n["total"],
            "N_ne": n["non_embedding"],
            "D": tokens,
            "loss": loss,
        }
    )
    if i == 0 or (i + 1) % 5 == 0 or i + 1 == len(plan):
        done = sum(r["D"] for r in runs) / planned_tokens
        eta = max(0.0, (time.time() - t_grid) / done * (1 - done) / 60)
        print(
            tr(
                f"  {i + 1}/{len(plan)} runs · ~{eta:.0f} min left",
                f"  {i + 1}/{len(plan)} تشغيلاً · بقي نحو {eta:.0f} دقيقة",
            )
        )

print()
print(
    tr(
        "   budget C     params N     tokens D    D/N    loss",
        "   الميزانية C   المعاملات N   الرموز D    D/N    الخسارة",
    )
)
for r in runs:
    print(
        f"  {r['C']:.1e}  {r['N']:>10,}  {r['D'] / 1e6:>8.2f}M  {r['D'] / r['N']:>6.1f}  {r['loss']:.4f}"
    )

n_runs = len(runs)
grid_minutes = round((time.time() - t_grid) / 60, 1)
worst_margin = round(min(unigram_loss - r["loss"] for r in runs), 3)
peak_vram_gb = 0.0
if device == "cuda":
    peak_vram_gb = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    print(
        tr(
            f"peak GPU memory {peak_vram_gb:.2f} GB",
            f"ذروة ذاكرة المعالج الرسومي {peak_vram_gb:.2f} غيغابايت",
        )
    )

_إعداد للرسوم فقط. وفي الدفتر العربي يحمّل أيضاً خطاً يحتوي الحروف العربية._

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import FuncFormatter, NullFormatter

if env.lang == "ar":
    # matplotlib draws Arabic as isolated letters in logical order. Reshape the
    # letters into their joined forms and reorder for display, and use a font
    # that actually contains Arabic glyphs (the default DejaVu does not).
    import arabic_reshaper
    from bidi.algorithm import get_display

    font_path = env.assets["NotoNaskhArabic-Regular.ttf"]
    font_manager.fontManager.addfont(str(font_path))
    arabic_font = font_manager.FontProperties(fname=str(font_path)).get_name()
    # Naskh has no Latin glyphs; DejaVu fills in digits, "C", "N" and symbols.
    plt.rcParams["font.family"] = [arabic_font, "DejaVu Sans"]
    plt.rcParams["mathtext.fontset"] = "dejavusans"
    plt.rcParams["axes.unicode_minus"] = False


def lab(en, ar):
    """A figure label in the notebook's language, ready for matplotlib."""
    if env.lang != "ar":
        return en
    return get_display(arabic_reshaper.reshape(ar))


def log_axis(axis):
    """Log scale with plain tick text such as 3e12, which renders in either font."""
    axis.set_major_formatter(
        FuncFormatter(lambda v, _: f"{v:.0e}".replace("e+0", "e").replace("e+", "e"))
    )
    axis.set_minor_formatter(NullFormatter())


budget_colors = plt.cm.viridis(np.linspace(0.1, 0.85, len(budgets)))
print(tr("figure text ready", "نصوص الرسوم جاهزة"))

_كل لون يمثّل ميزانية واحدة، والنجوم تحدّد قاع كل وادٍ. راقبها وهي تتحرّك نحو اليمين كلما زادت الحوسبة._

In [ ]:
def isoflop_optima(runs, count="total"):
    """Locate the best model size at each budget.

    Fit a parabola in log N to the lowest-loss run and up to two neighbours on
    each side; its vertex is the optimum. The optimum counts as measured only if
    the best run has a neighbour on both sides, so the minimum is bracketed.

    `count` chooses which parameter count goes on the x-axis. The training runs
    are the same either way; only the bookkeeping changes.
    """
    key = "N" if count == "total" else "N_ne"
    optima = []
    for C in budgets:
        group = sorted((r for r in runs if r["C"] == C), key=lambda r: r[key])
        if len(group) < 3:
            continue
        best = int(np.argmin([r["loss"] for r in group]))
        lo = max(0, min(best - 2, len(group) - 3))
        window = group[lo : max(lo + 3, best + 3)]
        x = np.log([r[key] for r in window])
        y = np.array([r["loss"] for r in window])
        a2, a1, a0 = np.polyfit(x, y, 2)
        x_star = -a1 / (2 * a2) if a2 > 0 else float("nan")
        interior = bool(0 < best < len(group) - 1 and a2 > 0 and x.min() < x_star < x.max())
        n_opt = math.exp(x_star) if a2 > 0 else float("nan")
        # Tokens at the optimum: D = C / (6 N_total). Under the non-embedding
        # convention, map N_ne back to N_total across the fitted runs first.
        xt = np.log([r["N"] for r in window])
        n_total_opt = math.exp(np.interp(x_star, x, xt)) if a2 > 0 else float("nan")
        d_opt = C / (6 * n_total_opt)
        optima.append(
            {
                "C": C,
                "C_count": 6 * n_opt * d_opt,
                "N_opt": n_opt,
                "D_opt": d_opt,
                "L_opt": a0 + a1 * x_star + a2 * x_star**2 if a2 > 0 else float("nan"),
                "interior": interior,
                "fit": (a2, a1, a0),
                "group": group,
                "window": (x.min(), x.max()),
            }
        )
    return optima


def frontier_points(optima):
    """The optima a frontier may be fitted to, and whether there are enough."""
    good = [o for o in optima if o["interior"]]
    if len(good) >= 3:
        return good, True
    return [o for o in optima if math.isfinite(o["N_opt"])], False


optima = isoflop_optima(runs)
bracketed_share = round(sum(o["interior"] for o in optima) / len(budgets), 2)

fig, ax = plt.subplots(figsize=(7.5, 4.6))
for o in optima:
    color = budget_colors[budgets.index(o["C"])]
    xs = np.array([r["N"] for r in o["group"]])
    ax.scatter(xs, [r["loss"] for r in o["group"]], color=color, s=28, zorder=3)
    grid_x = np.linspace(o["window"][0] - 0.1, o["window"][1] + 0.1, 100)
    ax.plot(
        np.exp(grid_x), np.polyval(o["fit"], grid_x), color=color, lw=1.4, label=f"C = {o['C']:.1e}"
    )
    if o["interior"]:
        ax.scatter(
            [o["N_opt"]],
            [o["L_opt"]],
            marker="*",
            s=160,
            color=color,
            edgecolor="black",
            linewidth=0.6,
            zorder=4,
        )
ax.set_xscale("log")
log_axis(ax.xaxis)
ax.set_xlabel(lab("parameters N (log scale)", "عدد المعاملات N (مقياس لوغاريتمي)"))
ax.set_ylabel(lab("validation loss (nats/byte)", "خسارة التحقق (نات لكل بايت)"))
ax.set_title(
    lab(
        "IsoFLOP curves: same compute, different splits",
        "منحنيات الحوسبة الثابتة: الحوسبة نفسها بتقسيمات مختلفة",
    )
)
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

for o in optima:
    if not math.isfinite(o["N_opt"]):
        print(
            f"C = {o['C']:.1e}:  "
            + tr(
                "no minimum: loss only falls or only rises across these sizes",
                "لا يوجد قاع: الخسارة تنخفض فقط أو ترتفع فقط عبر هذه الأحجام",
            )
        )
        continue
    flag = "" if o["interior"] else tr("  ← edge of the sampled range", "  ← على حافة المدى المُجرَّب")
    print(
        f"C = {o['C']:.1e}:  N_opt ≈ {o['N_opt'] / 1e3:,.0f}k   D_opt ≈ {o['D_opt'] / 1e6:.1f}M   "
        f"L ≈ {o['L_opt']:.4f}{flag}"
    )

كل منحنى يشبه وادياً. على يسار القاع نماذج أصغر من اللازم: قرأت بيانات وفيرة، لكن لم تكن لديها سعة كافية للاستفادة منها. وعلى يمين القاع نماذج أكبر من اللازم: لديها السعة، لكنّ الحوسبة نفدت قبل أن تقرأ بيانات كافية. النجمة هي نقطة التوازن <Term en="Compute-Optimal">الأمثل حوسبياً</Term> بين الحالتين.

لا تُحتسب النجمة إلا إذا وقعت بين تشغيلات حقيقية. يمكن دائماً مدّ القطع المكافئ إلى ما بعد آخر نقطة. لكنّ رأسه هناك تخمين، وليس قياساً. وهذا ما يحرسه فحص الحصر.

_ثلاثة توفيقات عبر النجوم: أفضل حجم، وأفضل كمية بيانات، وأفضل خسارة، كلٌّ منها مقابل الحوسبة._

In [ ]:
def power_fit(xs, ys):
    """Fit y = k · x^p in log space. Returns (p, k, r2)."""
    lx, ly = np.log(xs), np.log(ys)
    p, logk = np.polyfit(lx, ly, 1)
    resid = ly - (p * lx + logk)
    r2 = 1 - (resid**2).sum() / ((ly - ly.mean()) ** 2).sum()
    return float(p), float(math.exp(logk)), float(r2)


def loss_fit(cs, ls, with_floor=True):
    """Fit L(C) = E + A · C^(-alpha). With with_floor=False, E is fixed at zero.

    E is found by scanning: for each candidate floor, the rest is a straight line
    in log space, so the scan is exact and needs no optimizer.
    """
    cs, ls = np.asarray(cs), np.asarray(ls)
    floors = np.linspace(0.0, ls.min() * 0.995, 400) if with_floor else np.array([0.0])
    best = None
    for e in floors:
        slope, logA = np.polyfit(np.log(cs), np.log(ls - e), 1)
        sse = ((e + np.exp(logA) * cs**slope - ls) ** 2).sum()
        if best is None or sse < best[0]:
            best = (sse, float(e), float(math.exp(logA)), float(-slope))
    return best[1], best[2], best[3]  # E, A, alpha


good, enough_interior = frontier_points(optima)
if not enough_interior:
    # A minimum outside the sampled sizes is an extrapolation of a parabola, not
    # a measurement. Fall back so the cell still runs, but the check will fail.
    print(
        tr(
            "⚠ fewer than three budgets have a bracketed optimum; widen `ratios`",
            "⚠ أقلّ من ثلاث ميزانيات لها حجم أمثل محصور بين تجربتين؛ وسّع قائمة ratios",
        )
    )
C_fit = np.array([o["C"] for o in good])
exponent_a, k_N, frontier_r2 = power_fit(C_fit, [o["N_opt"] for o in good])
exponent_b, k_D, _ = power_fit(C_fit, [o["D_opt"] for o in good])
fitted_E, fit_A, alpha_c = loss_fit(C_fit, [o["L_opt"] for o in good])
tokens_per_param = round(good[-1]["D_opt"] / good[-1]["N_opt"], 1)
a_measured, E_frontier, frontier_r2 = (
    round(exponent_a, 3),
    round(fitted_E, 3),
    round(frontier_r2, 3),
)
if not enough_interior:
    frontier_r2 = 0.0  # a line through extrapolated minima is not a measured fit

print(
    tr(
        f"N_opt ∝ C^{exponent_a:.3f}   (R² {frontier_r2:.3f})\n"
        f"D_opt ∝ C^{exponent_b:.3f}\n"
        f"L_opt(C) = {fitted_E:.3f} + {fit_A:.3g} · C^-{alpha_c:.3f}\n"
        f"tokens per parameter at the largest fitted budget: {tokens_per_param:.1f}",
        f"N_opt ∝ C^{exponent_a:.3f}   (R² {frontier_r2:.3f})\n"
        f"D_opt ∝ C^{exponent_b:.3f}\n"
        f"L_opt(C) = {fitted_E:.3f} + {fit_A:.3g} · C^-{alpha_c:.3f}\n"
        f"عدد الرموز لكل معامل عند أكبر ميزانية: {tokens_per_param:.1f}",
    )
)
print(
    tr(
        "Chinchilla reported a ≈ 0.50 and roughly 20 tokens per parameter.",
        "للمقارنة: وجدت ورقة Chinchilla أنّ a ≈ 0.50، ونحو 20 رمزاً لكل معامل.",
    )
)

تقع النجوم قرب خط مستقيم على المقياس اللوغاريتمي. أي أنّ <bdi dir="ltr">N_opt ∝ C^a</bdi>، حيث <bdi dir="ltr">a = {{run.metrics.a_measured}}</bdi>. أما <bdi dir="ltr">Chinchilla</bdi> فقاست <bdi dir="ltr">a ≈ 0.5</bdi>. ومعنى ذلك أنّ المعاملات والبيانات يجب أن تنمو بالمعدّل نفسه.

لكنّ الرقم الأهمّ هو النسبة. عند أكبر ميزانية في التوفيق، يقرأ أفضل نموذج {{run.metrics.tokens_per_param}} رمزاً لكل معامل. رقم الـ20 في <bdi dir="ltr">Chinchilla</bdi> كان قياساً لنماذجهم ومُرمِّزهم وبياناتهم. ولا ينتقل هذا الرقم تلقائياً إلى إعداد آخر. أحد الأسباب المرجّحة هو وحدة القياس نفسها. فالبايت يحمل معلومات أقلّ بكثير من رمز المُرمِّز، الذي يغطّي عادةً عدّة أحرف. لذلك يحتاج نموذج البايتات إلى عدد أكبر منها. وسبب ثانٍ هو الحجم: عند بضع مئات الآلاف من المعاملات، تشكّل جداول التضمين جزءاً كبيراً من كل نموذج.

أما التوفيق الثالث، أي حدّ الخسارة <bdi dir="ltr">L_opt(C) = E + A·C^(−α)</bdi>، فيُظهر <Term en="Diminishing Returns">تناقص العوائد</Term>. كل مضاعفة للحوسبة تُزيل من الخسارة أقلّ مما أزالته المضاعفة التي قبلها. ولهذا ينحني المنحنى نحو أرضية <bdi dir="ltr">E = {{run.metrics.E_frontier}}</bdi> نات لكل بايت. هذه الأرضية هي <Term en="Irreducible Loss">الخسارة غير القابلة للاختزال</Term>.

القانون الذي لا يصف إلا النقاط التي وُفِّق عليها هو مجرّد <Term en="Interpolation (Statistical)">استيفاء</Term>. أما سبب توفيقه أصلاً فهو <Term en="Extrapolation (Statistical)">الاستقراء الخارجي</Term>، أي اختيار نموذج عند ميزانية لم تجرّبها بعد. لذلك تلتزم الخلية التالية، كتابةً، بحجم نموذج وبقيمة خسارة، لميزانية أكبر بـ{{scale.holdout_factor}} مرات من أيّ نقطة في التوفيق. وبعدها فقط تدرّبه الخلية التي تليها.

_لا تدريب في هذه الخلية. سجّل الخسارة المتوقَّعة قبل أن تشغّل الخلية التالية._

In [ ]:
# Commit to a number BEFORE training: the budget is beyond every fitted point.
C_holdout = budgets[-1] * cfg["holdout_factor"]
N_pred = k_N * C_holdout**exponent_a
L_pred = fitted_E + fit_A * C_holdout ** (-alpha_c)

# Use the shape closest to the predicted optimum that the data can feed.
candidates = sorted(ladder, key=lambda s: abs(math.log(sizes[s]["total"] / N_pred)))
holdout_shape = next(
    s for s in candidates if C_holdout / (6 * sizes[s]["total"]) <= max_train_tokens
)
holdout_params = sizes[holdout_shape]["total"]
holdout_tokens_m = C_holdout / (6 * holdout_params) / 1e6
predicted_loss = round(L_pred, 4)
holdout_tokens_m = round(holdout_tokens_m, 1)

print(
    tr(
        f"holdout budget {C_holdout:.1e} FLOPs ({cfg['holdout_factor']}× the largest fitted budget)\n"
        f"predicted optimum N ≈ {N_pred / 1e3:,.0f}k → nearest shape "
        f"{holdout_shape} with {holdout_params:,} params, {holdout_tokens_m:.1f}M tokens\n"
        f"predicted loss: {predicted_loss:.4f} nats/byte",
        f"ميزانية الاختبار {C_holdout:.1e} FLOPs (أي {cfg['holdout_factor']} أضعاف أكبر ميزانية في التوفيق)\n"
        f"الحجم الأمثل المتوقَّع N ≈ {N_pred / 1e3:,.0f}k ← أقرب حجم متاح "
        f"{holdout_shape} بعدد {holdout_params:,} معاملاً و{holdout_tokens_m:.1f} مليون رمز\n"
        f"الخسارة المتوقَّعة: {predicted_loss:.4f} نات لكل بايت",
    )
)

_المعيّن البرتقالي هو النقطة الوحيدة التي لم تشارك في التوفيق. لاحظ مدى قربه من الخط المتقطّع._

In [ ]:
t0 = time.time()
measured_loss, holdout_D = train_run(
    holdout_shape, C_holdout / (6 * holdout_params), cfg["seed"] + 999
)
prediction_error_pct = round(100 * abs(L_pred - measured_loss) / measured_loss, 2)
holdout_minutes = round((time.time() - t0) / 60, 1)

print(
    tr(
        f"predicted {predicted_loss:.4f}   measured {measured_loss:.4f}   "
        f"error {prediction_error_pct:.2f}%   ({holdout_minutes:.1f} min)",
        f"المتوقَّع {predicted_loss:.4f}   المقيس {measured_loss:.4f}   "
        f"الخطأ {prediction_error_pct:.2f}%   ({holdout_minutes:.1f} دقيقة)",
    )
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
cs = np.geomspace(C_fit.min() / 1.5, C_holdout * 1.5, 100)
ax1.scatter(
    C_fit,
    [o["N_opt"] for o in good],
    color="#3b5bdb",
    zorder=3,
    label=lab("IsoFLOP optima", "أمثل حجم لكل ميزانية"),
)
ax1.plot(
    cs, k_N * cs**exponent_a, color="#3b5bdb", lw=1.2, ls="--", label=f"N ∝ C^{exponent_a:.2f}"
)
ax1.scatter(
    [C_holdout],
    [holdout_params],
    marker="D",
    color="#e8590c",
    zorder=4,
    label=lab("holdout model", "نموذج الاختبار"),
)
ax1.set_xscale("log")
ax1.set_yscale("log")
log_axis(ax1.xaxis)
log_axis(ax1.yaxis)
ax1.set_xlabel(lab("compute C (FLOPs)", "الحوسبة C بوحدات FLOPs"))
ax1.set_ylabel(lab("compute-optimal N", "الحجم الأمثل N"))
ax1.legend(fontsize=8, frameon=False)

ax2.scatter(
    C_fit,
    [o["L_opt"] for o in good],
    color="#3b5bdb",
    zorder=3,
    label=lab("fitted points", "نقاط التوفيق"),
)
ax2.plot(
    cs,
    fitted_E + fit_A * cs ** (-alpha_c),
    color="#3b5bdb",
    lw=1.2,
    ls="--",
    label=lab("extrapolated frontier", "الحدّ المُستقرَأ"),
)
ax2.scatter(
    [C_holdout],
    [measured_loss],
    marker="D",
    color="#e8590c",
    zorder=4,
    label=lab("measured", "المقيس"),
)
ax2.axhline(fitted_E, color="grey", lw=0.8, ls=":")
ax2.set_xscale("log")
log_axis(ax2.xaxis)
ax2.set_xlabel(lab("compute C (FLOPs)", "الحوسبة C بوحدات FLOPs"))
ax2.set_ylabel(lab("loss (nats/byte)", "الخسارة (نات لكل بايت)"))
ax2.legend(fontsize=8, frameon=False)
fig.suptitle(
    lab(
        "Fitted on small budgets, tested on a larger one",
        "توفيق على ميزانيات صغيرة، واختبار على ميزانية أكبر",
    )
)
plt.tight_layout()
plt.show()

توقّع التوفيق {{run.metrics.predicted_loss}} نات لكل بايت. وقاس النموذج بعد تدريبه {{run.metrics.measured_loss}}، أي بفارق {{run.metrics.prediction_error_pct}}%.

هذا هو المنطق نفسه الذي حُدّد به حجم <bdi dir="ltr">Chinchilla</bdi> قبل تدريبها، لكن على مقياس أصغر بكثير. تدرّب نماذج صغيرة، ثم توفّق الحدّ الأمثل، ثم تستقرئ خارجه. وفي النهاية تفحص هذا الاستقراء بتشغيل واحد أكبر.

الطريقة الثالثة في <bdi dir="ltr">Chinchilla</bdi> تستخدم كل التشغيلات معاً، لا النجوم وحدها. فهي توفّق سطحاً واحداً: <bdi dir="ltr">L(N, D) = E + A/N^α + B/D^β</bdi>. ثم تقرأ منه أُسّ التوزيع: <bdi dir="ltr">a = β / (α + β)</bdi>. إذا كانت الطريقتان تصفان الواقع نفسه، فيجب أن تتقاربا. وفي الورقة تقاربتا فعلاً: 0.49 بطريقة الحوسبة الثابتة، و0.46 من السطح.

_قارن الأُسّين، وقارن الأرضيّتين. فكلها مأخوذة من التشغيلات نفسها._

In [ ]:
# Chinchilla's third approach: fit one surface L(N, D) = E + A/N^alpha + B/D^beta
# to every run at once, then read the allocation exponent off it.
import itertools


def fit_surface(runs):
    logN = torch.tensor([math.log(r["N"]) for r in runs], dtype=torch.float64)
    logD = torch.tensor([math.log(r["D"]) for r in runs], dtype=torch.float64)
    logL = torch.tensor([math.log(r["loss"]) for r in runs], dtype=torch.float64)

    def objective(th):
        # log L = log(exp(a - alpha·logN) + exp(b - beta·logD) + exp(e)), Huber in log space,
        # exactly as in the Chinchilla paper.
        pred = torch.logsumexp(
            torch.stack([th[0] - th[3] * logN, th[1] - th[4] * logD, th[2].expand_as(logN)]), dim=0
        )
        return F.huber_loss(pred, logL, delta=1e-3, reduction="sum")

    best = None
    # A small grid of starting points; the fit has several local minima.
    for a0, b0, e0, al0, be0 in itertools.product(
        (0.0, 5.0, 10.0), (0.0, 5.0, 10.0), (-1.0, 0.0, 0.5), (0.2, 0.5, 0.8), (0.2, 0.5, 0.8)
    ):
        th = torch.tensor([a0, b0, e0, al0, be0], dtype=torch.float64, requires_grad=True)
        opt = torch.optim.LBFGS([th], max_iter=200, line_search_fn="strong_wolfe")

        def closure(th=th, opt=opt):
            opt.zero_grad()
            loss = objective(th)
            loss.backward()
            return loss

        try:
            obj = opt.step(closure).item()
        except RuntimeError:
            continue
        if math.isfinite(obj) and (best is None or obj < best[0]):
            best = (obj, th.detach().clone())
    a, b, e, alpha, beta = best[1].tolist()
    return {"A": math.exp(a), "B": math.exp(b), "E": math.exp(e), "alpha": alpha, "beta": beta}


surface = fit_surface(runs)
param_exponent_a = round(surface["beta"] / (surface["alpha"] + surface["beta"]), 3)
param_E = round(surface["E"], 3)
surface_pred = round(
    surface["E"]
    + surface["A"] / holdout_params ** surface["alpha"]
    + surface["B"] / holdout_D ** surface["beta"],
    4,
)

print(
    tr(
        f"L(N, D) = {surface['E']:.3f} + {surface['A']:.3g}/N^{surface['alpha']:.3f} "
        f"+ {surface['B']:.3g}/D^{surface['beta']:.3f}\n"
        f"allocation exponent from the surface: a = β/(α+β) = {param_exponent_a:.3f}"
        f"   (IsoFLOP method: {exponent_a:.3f})\n"
        f"irreducible loss E: surface {param_E:.3f} · frontier {fitted_E:.3f}\n"
        f"holdout: surface predicts {surface_pred:.4f}, measured {measured_loss:.4f}",
        f"L(N, D) = {surface['E']:.3f} + {surface['A']:.3g}/N^{surface['alpha']:.3f} "
        f"+ {surface['B']:.3g}/D^{surface['beta']:.3f}\n"
        f"أُسّ التوزيع من السطح: a = β/(α+β) = {param_exponent_a:.3f}"
        f"   (بطريقة الحوسبة الثابتة: {exponent_a:.3f})\n"
        f"الخسارة غير القابلة للاختزال E: من السطح {param_E:.3f} · من الحدّ {fitted_E:.3f}\n"
        f"نموذج الاختبار: يتوقّع السطح {surface_pred:.4f}، والمقيس {measured_loss:.4f}",
    )
)

يعطي السطح <bdi dir="ltr">a = {{run.metrics.param_exponent_a}}</bdi> وأرضيةً <bdi dir="ltr">E = {{run.metrics.param_E}}</bdi>. وأعطت طريقة الحوسبة الثابتة <bdi dir="ltr">a = {{run.metrics.a_measured}}</bdi> و<bdi dir="ltr">E = {{run.metrics.E_frontier}}</bdi>. أياً كان الفرق بين الطريقتين، فهو ليس ضجيجاً في التدريب، لأنهما تقرآن التشغيلات نفسها. مصدر الفرق هو الافتراضات التي تبنيها كل طريقة. السطح يفترض أنّ صيغة رياضية واحدة تصحّ في كل مكان. أما طريقة الحوسبة الثابتة فتفترض فقط أنّ لكل وادٍ قاعاً قريباً من نجمته.

### تمرين — drop-the-floor

تعيد هذه الخلية توفيق حدّ الخسارة دون الخسارة غير القابلة للاختزال، أي كقانون قوة خالص. قبل أن تشغّلها، توقّع: هل سيضع خسارة نموذج الاختبار أعلى من الحقيقة أم أدنى منها؟ بعد ذلك قارن خطأه على نقاط التوفيق بخطئه على نموذج الاختبار. قد يبدو التوفيق مثالياً داخل مداه، ثم يخطئ خارجه.

_تلميح متاح: `env.hint(1)`_

In [ ]:
# YOUR TURN — the floor.
#
# The frontier fit above includes an irreducible loss E: a level no amount of
# compute gets below. This cell refits WITHOUT it, as a pure power law
# L = A · C^(-alpha), on exactly the same points. Before you run it, guess:
# will the pure power law predict a holdout loss that is too high or too low?
# Then set WITH_FLOOR = True and confirm you get the frontier fit back.
WITH_FLOOR = False

E_try, A_try, alpha_try = loss_fit(C_fit, [o["L_opt"] for o in good], with_floor=WITH_FLOOR)
floor_pred = round(E_try + A_try * C_holdout ** (-alpha_try), 4)
floor_error_pct = round(100 * (floor_pred - measured_loss) / measured_loss, 2)

in_range = np.array([E_try + A_try * c ** (-alpha_try) for c in C_fit])
in_range_rmse = float(np.sqrt(((in_range - np.array([o["L_opt"] for o in good])) ** 2).mean()))
print(
    tr(
        f"with_floor={WITH_FLOOR}: E={E_try:.3f}, alpha={alpha_try:.3f}\n"
        f"  error on the fitted points (RMSE): {in_range_rmse:.4f}\n"
        f"  holdout prediction {floor_pred:.4f} vs measured {measured_loss:.4f}"
        f"  → signed error {floor_error_pct:+.2f}%",
        f"with_floor={WITH_FLOOR}: E={E_try:.3f}, alpha={alpha_try:.3f}\n"
        f"  الخطأ على نقاط التوفيق (RMSE): {in_range_rmse:.4f}\n"
        f"  التوقّع لنموذج الاختبار {floor_pred:.4f} مقابل المقيس {measured_loss:.4f}"
        f"  ← الخطأ بإشارته {floor_error_pct:+.2f}%",
    )
)

### تمرين — count-like-kaplan

تعيد هذه الخلية توفيق الحدّ الأمثل باستخدام عدد المعاملات دون جداول التضمين. لا يُعاد تدريب أيّ شيء، والذي يتغيّر هو الرقم المكتوب بجانب كل نموذج فقط. قبل أن تشغّلها، خمّن: هل يرتفع الأُسّ <bdi dir="ltr">a</bdi> أم ينخفض؟ ثم قارنه بالقيمة التي نشرها <bdi dir="ltr">Kaplan</bdi>، أي 0.73، وبقيمة <bdi dir="ltr">Chinchilla</bdi>، أي 0.50.

_تلميح متاح: `env.hint(3)`_

In [ ]:
# YOUR TURN — the bookkeeping.
#
# Kaplan et al. counted parameters WITHOUT the embedding tables; Chinchilla
# counted all of them. This cell refits the frontier with the Kaplan count,
# from the very same training runs. Nothing is retrained. Only the number you
# write next to each model changes. Set COUNT = "total" to get the original back.
COUNT = "non_embedding"

recount, _ = frontier_points(isoflop_optima(runs, count=COUNT))
count_exponent_a, _, count_r2 = power_fit(
    [o["C_count"] for o in recount], [o["N_opt"] for o in recount]
)
count_exponent_a = round(count_exponent_a, 3)
count_tokens_per_param = round(recount[-1]["D_opt"] / recount[-1]["N_opt"], 1)
print(
    tr(
        f"count={COUNT}:  N_opt ∝ C^{count_exponent_a:.3f}  (R² {count_r2:.3f})   "
        f"tokens/param at the largest budget {count_tokens_per_param:.1f}\n"
        f"count=total:  N_opt ∝ C^{exponent_a:.3f}   tokens/param {tokens_per_param:.1f}",
        f"count={COUNT}:  N_opt ∝ C^{count_exponent_a:.3f}  (R² {count_r2:.3f})   "
        f"رموز لكل معامل عند أكبر ميزانية {count_tokens_per_param:.1f}\n"
        f"count=total:  N_opt ∝ C^{exponent_a:.3f}   رموز لكل معامل {tokens_per_param:.1f}",
    )
)

اختلف <bdi dir="ltr">Kaplan</bdi> و<bdi dir="ltr">Chinchilla</bdi> اختلافاً كبيراً في جواب السؤال نفسه. وقد أرجعت أعمال لاحقة جزءاً كبيراً من هذا الفرق إلى اختيارات تشبه اختيار هذا التمرين. منها عدّ المعاملات دون جداول التضمين. ومنها القياس عند أحجام صغيرة تهيمن فيها هذه الجداول. ومنها جداول معدّل تعلّم لا تطابق طول كل تشغيل. ما نستخلصه أنّ <Term en="Scaling Law">قانون التوسّع</Term> خاصيةٌ لإعداد القياس كله: النماذج، والبيانات، والمُرمِّز، والمُحسِّن، وطريقة العدّ. فإذا غيّرت واحداً منها، فعليك أن تقيس من جديد.

> **الورقة** · [language-models-are-few-shot-learners](https://azimuth.plus/ar/paper/language-models-are-few-shot-learners) — نموذج حُدّد حجمه بالقاعدة الأقدم، ثم تبيّن لاحقاً، بمقياس الحدّ الأمثل حوسبياً، أنه لم يُدرَّب بما يكفي
>
> درّب <bdi dir="ltr">GPT-3</bdi> نموذجاً فيه 175 مليار معامل على نحو 300 مليار رمز، أي أقلّ من رمزين لكل معامل. وبحسب قياس <bdi dir="ltr">Chinchilla</bdi> اللاحق، كان النموذج أكبر بكثير مما تبرّره ميزانية حوسبته.

_أول فحصين يتأكّدان من أنّ الشبكة قاست شيئاً فعلاً. والفحصان الأخيران يقيّمان ما فعلته بهذا القياس._

In [ ]:
learned_ok = env.check("runs-learned", worst_margin)
bracketed_ok = env.check("optima-bracketed", bracketed_share)
frontier_ok = env.check("frontier-fit", frontier_r2)
prediction_ok = env.check("prediction-error", prediction_error_pct)

In [ ]:
receipt = env.receipt()

ما يجب أن يبقى معك هو الإجراء، لا أُسٌّ بعينه. ثبّت الحوسبة، ونوّع التقسيم، وجِد قاع كل وادٍ. ثم مرّر خطاً عبر القيعان، واختبره عند ميزانية لم تدخل في التوفيق. بهذا الإجراء نفسه خُطّط لنماذج فيها مليارات المعاملات. وعلى تشغيلاتك أنت، أعطى {{run.metrics.tokens_per_param}} رمزاً لكل معامل بدلاً من 20. هذا الفرق ليس خطأً. إنه ما يبدو عليه قانون مقيس حين يتغيّر الإعداد.